In [0]:
from pyspark.sql import functions as F

CATALOG_SCHEMA = "workspace.ecommerce_de"

silver_orders = spark.table(f"{CATALOG_SCHEMA}.silver_orders")
silver_payments = spark.table(f"{CATALOG_SCHEMA}.silver_payments")
dim_customers_silver = spark.table(f"{CATALOG_SCHEMA}.dim_customers_silver")

# 1. Dimension: dim_customer (SCD Type 2 preserved)
dim_customer_df = (
    dim_customers_silver.withColumn(
        "customer_sk",
        F.sha2(
            F.concat_ws(
                "||",
                F.col("customer_id"),
                F.coalesce(F.col("effective_from").cast("string"), F.lit("INIT")),
            ),
            256,
        ),
    )
    .select(
        "customer_sk",
        "customer_id",
        "city",
        "loyalty_tier",
        "effective_from",
        "effective_to",
        "is_current",
        "silver_processed_at",
    )
    .withColumn("gold_updated_at", F.current_timestamp())
)

dim_customer_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{CATALOG_SCHEMA}.dim_customer"
)

# 2. Dimension: dim_date (Derived from orders & payments timestamps)
order_dates = silver_orders.select(F.to_date("created_at_ts").alias("dt"))
payment_dates = silver_payments.select(F.to_date("event_ts").alias("dt"))

dim_date_df = (
    order_dates.union(payment_dates)
    .filter(F.col("dt").isNotNull())
    .distinct()
    .withColumn("date_key", F.date_format("dt", "yyyyMMdd").cast("int"))
    .withColumn("year", F.year("dt"))
    .withColumn("month", F.month("dt"))
    .withColumn("month_name", F.date_format("dt", "MMM"))
    .withColumn("day_of_month", F.dayofmonth("dt"))
    .withColumn("day_of_week", F.dayofweek("dt"))
    .withColumn("is_weekend", F.col("day_of_week").isin(1, 7))
    .orderBy("dt")
)

dim_date_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{CATALOG_SCHEMA}.dim_date"
)

# 3. Fact: fact_orders (Point-in-time join with active/historical customer attribute)
fact_orders_df = (
    silver_orders.withColumn("order_date", F.to_date("created_at_ts"))
    .withColumn("order_date_key", F.date_format("created_at_ts", "yyyyMMdd").cast("int"))
    .alias("o")
    .join(
        dim_customer_df.alias("c"),
        (F.col("o.customer_id") == F.col("c.customer_id"))
        & (F.col("c.is_current") == True),
        how="left",
    )
    .select(
        F.col("o.order_id").alias("order_id"),
        F.col("o.customer_id").alias("customer_id"),
        F.col("c.customer_sk").alias("current_customer_sk"),
        F.col("o.order_date").alias("order_date"),
        F.col("o.order_date_key").alias("order_date_key"),
        F.col("o.created_at_ts").alias("created_at_ts"),
        F.col("o.status").alias("status"),
        F.col("o.order_amount").alias("order_amount"),
        F.col("o.batch_id").alias("batch_id"),
        F.current_timestamp().alias("gold_updated_at"),
    )
)

fact_orders_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{CATALOG_SCHEMA}.fact_orders"
)

# 4. Fact: fact_payments
fact_payments_df = (
    silver_payments.withColumn("payment_date", F.to_date("event_ts"))
    .withColumn("payment_date_key", F.date_format("event_ts", "yyyyMMdd").cast("int"))
    .select(
        "event_id",
        "order_id",
        "payment_date",
        "payment_date_key",
        "event_ts",
        "payment_method",
        "payment_status",
        "payment_amount",
        "batch_id",
        F.current_timestamp().alias("gold_updated_at"),
    )
)

fact_payments_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{CATALOG_SCHEMA}.fact_payments"
)

print("✅ Star schema tables (dim_customer, dim_date, fact_orders, fact_payments) created.")

In [0]:
%sql
CREATE OR REPLACE VIEW workspace.ecommerce_de.mart_daily_revenue AS
WITH order_agg AS (
  SELECT
    order_date,
    COUNT(DISTINCT order_id) AS total_orders,
    COUNT(DISTINCT CASE WHEN status = 'COMPLETED' THEN order_id END) AS completed_orders,
    ROUND(SUM(CASE WHEN status IN ('COMPLETED', 'PENDING') THEN order_amount ELSE 0 END), 2) AS gross_order_value,
    ROUND(SUM(CASE WHEN status = 'REFUNDED' THEN order_amount ELSE 0 END), 2) AS refunded_order_value
  FROM workspace.ecommerce_de.fact_orders
  WHERE order_date IS NOT NULL
  GROUP BY order_date
),
payment_agg AS (
  SELECT
    payment_date,
    ROUND(SUM(CASE WHEN UPPER(payment_status) IN ('SUCCESS', 'COMPLETED', 'PAID') THEN payment_amount ELSE 0 END), 2) AS settled_payment_amount,
    COUNT(CASE WHEN UPPER(payment_status) IN ('FAILED', 'DECLINED', 'ERROR') THEN 1 END) AS failed_payment_events
  FROM workspace.ecommerce_de.fact_payments
  WHERE payment_date IS NOT NULL
  GROUP BY payment_date
)
SELECT
  COALESCE(o.order_date, p.payment_date) AS metric_date,
  COALESCE(o.total_orders, 0) AS total_orders,
  COALESCE(o.completed_orders, 0) AS completed_orders,
  COALESCE(o.gross_order_value, 0.0) AS gross_order_value,
  COALESCE(o.refunded_order_value, 0.0) AS refunded_order_value,
  COALESCE(p.settled_payment_amount, 0.0) AS settled_payment_amount,
  COALESCE(p.failed_payment_events, 0) AS failed_payment_events
FROM order_agg o
FULL OUTER JOIN payment_agg p
  ON o.order_date = p.payment_date
ORDER BY metric_date DESC;

CREATE OR REPLACE VIEW workspace.ecommerce_de.mart_customer_360 AS
WITH cust_orders AS (
  SELECT
    customer_id,
    COUNT(DISTINCT order_id) AS total_orders,
    ROUND(SUM(CASE WHEN status != 'CANCELLED' THEN order_amount ELSE 0 END), 2) AS lifetime_spend,
    MAX(order_date) AS last_order_date,
    DATEDIFF(CURRENT_DATE(), MAX(order_date)) AS days_since_last_order
  FROM workspace.ecommerce_de.fact_orders
  GROUP BY customer_id
)
SELECT
  c.customer_id,
  c.city,
  c.loyalty_tier,
  COALESCE(o.total_orders, 0) AS total_orders,
  COALESCE(o.lifetime_spend, 0.0) AS lifetime_spend,
  o.last_order_date,
  o.days_since_last_order,
  CASE
    WHEN o.last_order_date IS NULL THEN 'NEVER_ORDERED'
    WHEN o.days_since_last_order > 60 AND COALESCE(o.total_orders, 0) >= 2 THEN 'HIGH_CHURN_RISK'
    WHEN o.days_since_last_order > 30 THEN 'MEDIUM_CHURN_RISK'
    ELSE 'ACTIVE'
  END AS retention_segment
FROM workspace.ecommerce_de.dim_customer c
LEFT JOIN cust_orders o
  ON c.customer_id = o.customer_id
WHERE c.is_current = TRUE;

In [0]:
%sql
SELECT * FROM workspace.ecommerce_de.mart_daily_revenue LIMIT 10;